In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/zalando-research/fashionmnist/t10k-labels-idx1-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/t10k-images-idx3-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_test.csv
/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_train.csv
/kaggle/input/datasets/zalando-research/fashionmnist/train-labels-idx1-ubyte
/kaggle/input/datasets/zalando-research/fashionmnist/train-images-idx3-ubyte


In [2]:
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
torch.manual_seed(42)

In [4]:
df = pd.read_csv("/kaggle/input/datasets/zalando-research/fashionmnist/fashion-mnist_train.csv")


In [5]:
x = df.iloc[:, 1 : ]. values
y = df.iloc[:, 0].values

In [6]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [7]:
X_train = X_train/255.0
X_test = X_test/255.0

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [9]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype = torch.float32)
        self.labels = torch.tensor(labels, dtype = torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [10]:
train_dataset = CustomDataset(X_train, y_train)
len(train_dataset)

48000

In [11]:
test_dataset = CustomDataset(X_test, y_test)

In [12]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)

In [13]:
class MyNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)

In [14]:
EPOCHS = 100
LEARNING_RATE = 0.1

In [15]:
model = MyNN(X_train.shape[1])
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = LEARNING_RATE)

In [16]:
len(train_loader)

1500

In [17]:
for epoch in range(EPOCHS):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch:{epoch+1}, Loss:{avg_loss}')

Epoch:1, Loss:0.6352872474888961
Epoch:2, Loss:0.4304986953884363
Epoch:3, Loss:0.3861262078657746
Epoch:4, Loss:0.3584607255011797
Epoch:5, Loss:0.3376494748592377
Epoch:6, Loss:0.32276468626906474
Epoch:7, Loss:0.3078539018382629
Epoch:8, Loss:0.2949818898836772
Epoch:9, Loss:0.2854692505300045
Epoch:10, Loss:0.27467058210571604
Epoch:11, Loss:0.26830569267148774
Epoch:12, Loss:0.2581421597401301
Epoch:13, Loss:0.24940819991752505
Epoch:14, Loss:0.24444738873218497
Epoch:15, Loss:0.2385919222868979
Epoch:16, Loss:0.23155899402375021
Epoch:17, Loss:0.22562562982489665
Epoch:18, Loss:0.2202964697740972
Epoch:19, Loss:0.21206334652379155
Epoch:20, Loss:0.20960091769819458
Epoch:21, Loss:0.20624992956593632
Epoch:22, Loss:0.19986103242821993
Epoch:23, Loss:0.1953041393881043
Epoch:24, Loss:0.19312163671137145
Epoch:25, Loss:0.18764107141271233
Epoch:26, Loss:0.18366442496639987
Epoch:27, Loss:0.18018299543857574
Epoch:28, Loss:0.17326497477230926
Epoch:29, Loss:0.17048049322267372
Epoch:

In [18]:
torch.save(model.state_dict(), '/kaggle/working/my_trained_model.pth')


In [19]:
model.eval()
total = 0
correct = 0
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)
        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8898333333333334
